# YOLO Foundations Part 2: Real-World Solutions

**Format:** Station-based Exercises + Group Capstone

In this lab, we'll explore **Ultralytics Solutions** — high-level wrappers that transform raw detections into actionable business insights.

## Learning Objectives
- Apply region-based counting with custom polygons
- Measure speed and distance between tracked objects
- Generate analytics charts and heatmaps from video data
- Implement privacy-compliant object blurring
- Combine multiple solutions into a complete pipeline

---

## Video Sources

We'll use sample videos from local files and GitHub:

| Source | File | Use Case |
|--------|------|----------|
| **Local** | `Pull_ups.mp4` | Workout monitoring |
| **Local** | `parking_slots.mp4` | Parking management |
| **Local** | `Cars.mp4` | Sample traffic video |

---

## 1. Environment Setup

In [ ]:
# Install Ultralytics and import libraries
import sys
%pip install -q ultralytics opencv-python pandas matplotlib requests sahi
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import os
import requests
from ultralytics import YOLO, solutions

# Download videos from GitHub if not exists locally
videos = ["Pull_ups.mp4", "parking_slots.mp4", "Cars.mp4"]
# Updated loop logic
for video_file in videos:
    if not os.path.exists(video_file):
        # Using the direct download link structure
        url = f"https://raw.githubusercontent.com/NaifMersal/cv-for-developers-ultralytics/main/slides/assets/{video_file}"
        
        # Adding stream=True is better for large video files
        r = requests.get(url, stream=True)
        
        if r.status_code == 200:
            with open(video_file, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"✓ Downloaded: {video_file}")
        else:
            print(f"✗ Failed to download {video_file}. Status code: {r.status_code}")

---

# Station 1: Region & Boundary Analytics (40 min)

## 📍 Exercise 1.1: Custom Counting Region (15 min)

**Task:** Count vehicles crossing a specific lane using a custom polygon region. Compare results between a line and a polygon.

**Video:** Urban traffic intersection

In [ ]:
video_path = "Cars.mp4"

# Download if not exists
if not os.path.exists(video_path):
    url = f"https://raw.githubusercontent.com/NaifMersal/cv-for-developers-ultralytics/main/slides/assets/{video_path}"
    r = requests.get(url)
    with open(video_path, "wb") as f:
        f.write(r.content)

# Display the video
Video(video_path, width=640, height=360, embed=True)

In [ ]:
# 🚗 Exercise 1.1: Object Counting with Custom Region

import cv2
import os
from ultralytics import YOLO, solutions
from IPython.display import Video

# 1. Setup
temp_output = "temp_raw.mp4"
final_output = "lane_counting_5fps.mp4"
model = YOLO("weights/yolo26n.pt")

cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

# 2. Sampling Logic
TARGET_FPS = 5
skip_rate = max(1, int(original_fps / TARGET_FPS))
print(f"Original FPS: {original_fps} | Sampling every {skip_rate} frames")

# 3. Define Polygon & Initialize Counter
lane_region = [(int(w*0.4), int(h*0.6)), (int(w*0.6), int(h*0.6)),
               (int(w*0.9), int(h*0.9)), (int(w*0.1), int(h*0.9))]

lane_region = [(int(w*0.4), int(h*0.6)), (int(w*0.6), int(h*0.6))]

counter = solutions.ObjectCounter(
    region=lane_region,
    model=model,
    classes=[2, 3, 5, 7],
    show=False
)

# 4. Video Writer set to 5 FPS
writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))

frame_idx = 0
while cap.isOpened():
    success, im0 = cap.read()
    if not success: break

    # Only process frames that match our 5 FPS target
    if frame_idx % skip_rate == 0:
        results = counter(im0)

        # Access the annotated image from the results
        annotated_frame = results.plot_im
        writer.write(annotated_frame)

    frame_idx += 1
    if frame_idx > 300: break # Stop after ~10 seconds of source video

cap.release()
writer.release()

# 5. Notebook Display Fix
if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

print(f"📊 Processed {frame_idx // skip_rate} frames. IN: {counter.in_count} | OUT: {counter.out_count}")

Video(final_output, embed=True, width=800)

### 🏆 Challenge (Optional)
Try different region shapes:
- A horizontal line across half the frame
- A rectangle covering only the left lane
- Which gives more accurate counts?

---

## 🅿️ Exercise 1.2: Parking Lot Management (15 min)

**Task:** Define parking spots and track occupancy over time. Log results to CSV.

**Video:** Drone footage of mall parking lot

In [ ]:
import os
import requests
video_path = "parking_slots.mp4"
# Download if not exists
if not os.path.exists(video_path):
    url = "https://raw.githubusercontent.com/NaifMersal/cv-for-developers-ultralytics/main/slides/assets/parking_slots.mp4"
    r = requests.get(url)
    with open(video_path, "wb") as f:
        f.write(r.content)

### 🏆 Challenge
Add a real-time alert when occupancy exceeds 75%

---

## 📋 Exercise 1.3: Queue Alert System (10 min)

**Task:** Monitor a region and trigger alert when queue exceeds threshold.

In [ ]:
# 📋 Exercise 1.3: Queue Alert System

import cv2
import os
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from ultralytics import YOLO, solutions
from shapely.geometry import Point, Polygon
import numpy as np

# 1. Setup
temp_output = "sahi_queue_raw.mp4"
final_output = "sahi_queue_5fps.mp4"
video_path = "Cars.mp4"

# Load Model using SAHI's AutoDetectionModel as per Docs
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="weights/yolo26n.pt", # Use your specific weights
    confidence_threshold=0.3,
    device="cpu", # or "cuda"
)

cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

# 2. Sampling Logic
TARGET_FPS = 3
skip_rate = max(1, int(original_fps / TARGET_FPS))

# 3. Define Queue Region (Polygon)
queue_points = [(int(w*0.3), int(h*0.5)), (int(w*0.7), int(h*0.5)),
                (int(w*0.9), int(h*0.9)), (int(w*0.1), int(h*0.9))]
queue_polygon = Polygon(queue_points)

# 4. Processing Loop
writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))

frame_idx = 0
while cap.isOpened():
    success, frame = cap.read()
    if not success: break

    # Only process frames for the 5 FPS target
    if frame_idx % skip_rate == 0:
        # Perform Sliced Inference (SAHI)
        # This solves the "No tracks found" by detecting small objects in 4K
        result = get_sliced_prediction(
            frame,
            detection_model,
            slice_height=640,
            slice_width=640,
            overlap_height_ratio=0.2,
            overlap_width_ratio=0.2
        )

        occupancy_count = 0
        annotated_frame = frame.copy()

        # Handle Prediction Results (from SAHI Docs)
        for object_prediction in result.object_prediction_list:
            bbox = object_prediction.bbox.to_xyxy() # [x1, y1, x2, y2]
            cls_id = object_prediction.category.id

            # Filter for cars/trucks (class 2, 7)
            if cls_id in [2, 7]:
                # Calculate center point of the detection
                cx, cy = int((bbox[0] + bbox[2]) / 2), int((bbox[1] + bbox[3]) / 2)

                # Check if object is inside the Queue Region
                if queue_polygon.contains(Point(cx, cy)):
                    occupancy_count += 1
                    color = (0, 255, 0) # Green for inside queue
                else:
                    color = (255, 0, 0) # Blue for outside

                # Draw bounding box and center point
                cv2.rectangle(annotated_frame, (int(bbox[0]), int(bbox[1])),
                              (int(bbox[2]), int(bbox[3])), color, 2)
                cv2.circle(annotated_frame, (cx, cy), 5, color, -1)

        # Draw the Queue Region Polygon
        cv2.polylines(annotated_frame, [np.array(queue_points)], True, (255, 255, 0), 3)
        cv2.putText(annotated_frame, f"Queue Occupancy: {occupancy_count}", (50, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 255), 3)

        writer.write(annotated_frame)

    frame_idx += 1
    if frame_idx > 300: break

cap.release()
writer.release()

# 5. Finalize Video
if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

Video(final_output, embed=True, width=800)

---

# Station 2: Kinematic & Spatial Measurement (35 min)

## 🚀 Exercise 2.1: Speed Estimation (20 min)

**Task:** Estimate vehicle speeds by calibrating `meter_per_pixel`. Identify the fastest vehicle.

**Hint:** Start with `meter_per_pixel=0.05` and adjust based on visual reference.

In [ ]:
# 🚀 Exercise 2.1: Speed Estimation

video_path = "Cars.mp4"
temp_output = "speed_raw.mp4"
final_output = "speed_5fps.mp4"

model = YOLO("weights/yolo26n.pt")
cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

TARGET_FPS = 5
skip_rate = max(1, int(original_fps / TARGET_FPS))

METER_PER_PIXEL = 0.05  # 🔧 ADJUST THIS

speed_est = solutions.SpeedEstimator(
    show=False,
    model=model,
    fps=TARGET_FPS,
    meter_per_pixel=METER_PER_PIXEL,
    classes=[2, 3, 5, 7]
)

writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))

speed_readings = []
frame_idx = 0
while cap.isOpened():
    success, im0 = cap.read()
    if not success: break
    
    if frame_idx % skip_rate == 0:
        results = speed_est(im0)
        writer.write(results.plot_im)
        
        # Access speed data
        if hasattr(results, 'speed') and results.speed is not None:
            for track_id, speed in results.speed.items():
                if speed > 0:
                    speed_readings.append({'frame': frame_idx, 'track_id': track_id, 'speed_kmh': speed})
    
    frame_idx += 1
    if frame_idx > 200: break

cap.release()
writer.release()

if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

# Summary stats
if speed_readings:
    df = pd.DataFrame(speed_readings)
    print(f"\n📊 Avg speed: {df['speed_kmh'].mean():.1f} km/h | Max speed: {df['speed_kmh'].max():.1f} km/h")
else:
    print("⚠️ No speed data captured")

from IPython.display import Video
Video(final_output, embed=True, width=800)

### 🏆 Challenge
Adjust `meter_per_pixel` to get more realistic speeds. What value works best?

---

## 📏 Exercise 2.2: Distance Calculation (15 min)

**Task:** Calculate distance between two tracked vehicles. Trigger warning if too close.

In [ ]:
# 📏 Exercise 2.2: Distance Calculation

video_path = "Cars.mp4"
temp_output = "distance_raw.mp4"
final_output = "distance_5fps.mp4"

model = YOLO("weights/yolo26n.pt")
cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

TARGET_FPS = 5
skip_rate = max(1, int(original_fps / TARGET_FPS))

dist_calc = solutions.DistanceCalculation(
    show=False,
    model=model,
    classes=[2, 3, 5, 7]
)

writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))
frame_idx = 0
while cap.isOpened():
    success, im0 = cap.read()
    if not success: break
    
    if frame_idx % skip_rate == 0:
        results = dist_calc(im0)
        writer.write(results.plot_im)
    
    frame_idx += 1
    if frame_idx > 150: break

cap.release()
writer.release()

if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

from IPython.display import Video
Video(final_output, embed=True, width=800)

---

# Station 3: Privacy, Analytics & Visualization (35 min)

## 🔒 Exercise 3.1: Object Blurring (10 min)

**Task:** Blur detected persons for privacy compliance.

**Challenge:** Try to blur only faces using pose keypoints (advanced).

In [ ]:
import os
import requests
video_path = "parking_slots.mp4"
# Download if not exists
if not os.path.exists(video_path):
    url = "https://raw.githubusercontent.com/NaifMersal/cv-for-developers-ultralytics/main/slides/assets/parking_slots.mp4"
    r = requests.get(url)
    with open(video_path, "wb") as f:
        f.write(r.content)

### 🏆 Advanced Challenge: Face-Only Blur
Use YOLO Pose (`yolov8n-pose.pt`) and blur only keypoints 0-4 (face)

---

## 📈 Exercise 3.2: Analytics Dashboard (15 min)

**Task:** Generate line/bar/pie charts showing object class distribution over time.

In [ ]:
# 📈 Exercise 3.2: Analytics Dashboard

video_path = "Cars.mp4"
temp_output = "analytics_raw.mp4"
final_output = "analytics_5fps.mp4"

model = YOLO("weights/yolo26n.pt")
cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

TARGET_FPS = 5
skip_rate = max(1, int(original_fps / TARGET_FPS))

analytics = solutions.Analytics(
    show=False,
    analytics_type="line",
    model=model,
    classes=[2, 3, 5, 7]
)

writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))
frame_idx = 0
while cap.isOpened():
    success, im0 = cap.read()
    if not success: break
    
    if frame_idx % skip_rate == 0:
        results = analytics(im0, frame_count=frame_idx)
        writer.write(results.plot_im)
    
    frame_idx += 1
    if frame_idx > 200: break

cap.release()
writer.release()

if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

from IPython.display import Video
Video(final_output, embed=True, width=800)

### 🏆 Challenge
Create all 4 chart types (line, bar, pie, area) and identify peak hours

---

## 🔥 Exercise 3.3: Heatmap Visualization (10 min)

**Task:** Generate a heatmap showing movement density. Compare colormaps.

In [ ]:
import os
import requests
video_path = "parking_slots.mp4"
# Download if not exists
if not os.path.exists(video_path):
    url = "https://raw.githubusercontent.com/NaifMersal/cv-for-developers-ultralytics/main/slides/assets/parking_slots.mp4"
    r = requests.get(url)
    with open(video_path, "wb") as f:
        f.write(r.content)

---

# Station 4: Specialized Heads (30 min)

## 🏋️ Exercise 4.1: Workout Monitoring (15 min)

**Task:** Count pushups using YOLO Pose. Modify for squats or jumping jacks.

**Video:** Person doing pull-ups

In [ ]:
# 🏋️ Exercise 4.1: Workout Monitoring

video_path = "Pull_ups.mp4"
temp_output = "gym_raw.mp4"
final_output = "gym_5fps.mp4"

model = YOLO("yolov8n-pose.pt")
cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv2.CAP_PROP_FPS)

TARGET_FPS = 5
skip_rate = max(1, int(original_fps / TARGET_FPS))

gym = solutions.AIGym(
    show=False,
    kpts=[6, 8, 10],  # shoulder-elbow-wrist
    model=model
)

writer = cv2.VideoWriter(temp_output, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))
frame_idx = 0
while cap.isOpened():
    success, im0 = cap.read()
    if not success: break
    
    if frame_idx % skip_rate == 0:
        results = gym(im0)
        writer.write(results.plot_im)
    
    frame_idx += 1
    if frame_idx > 200: break

cap.release()
writer.release()

if os.path.exists(temp_output):
    os.system(f"ffmpeg -i {temp_output} -vcodec libx264 -f mp4 {final_output} -y -loglevel quiet")

print(f"💪 Rep count: {getattr(gym, 'count', 'N/A')}")

from IPython.display import Video
Video(final_output, embed=True, width=800)

### 🏆 Challenge
Modify the keypoints for:
- **Squats:** kpts=[11, 13, 15] (hip-knee-ankle)
- **Jumping jacks:** kpts=[5, 7, 9] + [6, 8, 10] (both arms)

---

## 🎭 Exercise 4.2: Instance Segmentation Tracking (15 min)

**Task:** Track objects with pixel-perfect masks. Compare to bounding box tracking.

In [ ]:
# 🎭 Exercise 4.2: Instance Segmentation Tracking

video_path = "Cars.mp4"

# Use YOLO Segmentation model
model = YOLO("yolo26n-seg.pt")
cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Error loading video"

w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

frame_count = 0
max_frames = 50

# Track using segmentation model
results = model.track(source=video_path, stream=True, classes=[2, 3, 5, 7], persist=True)

tracked_objects = set()

for result in results:
    if result.boxes.id is not None:
        for track_id in result.boxes.id.cpu().numpy():
            tracked_objects.add(int(track_id))
    frame_count += 1
    if frame_count >= max_frames:
        break

print(f"✓ Tracked {len(tracked_objects)} unique vehicles with instance masks")

---

# Capstone Project: Smart Retail Analytics System (60 min)

## 🎯 Group Challenge (3-4 students)

Build a complete retail analytics system that:

1. **Counts customers** entering the store (ObjectCounter)
2. **Generates heatmap** of popular aisles (Heatmap)
3. **Blurs faces** for privacy (ObjectBlurrer)
4. **Shows peak hours** with analytics chart (Analytics)
5. **Alerts** if checkout queue > 3 people (Queue alert)

### Requirements
- Use **at least 3 different solutions**
- Process video end-to-end
- Export results to CSV/JSON

### Stretch Goals
- Add speed estimation for customer walking patterns
- Create a simple dashboard visualization
- Combine multiple videos

### Deliverables
- Working notebook with all solutions integrated
- `retail_summary.csv` with counts, peak times, occupancy
- 2-minute demo per group

In [ ]:
import os
import requests
video_path = "parking_slots.mp4"
# Download if not exists
if not os.path.exists(video_path):
    url = "https://raw.githubusercontent.com/NaifMersal/cv-for-developers-ultralytics/main/slides/assets/parking_slots.mp4"
    r = requests.get(url)
    with open(video_path, "wb") as f:
        f.write(r.content)

---

## ✅ Wrap-up Questions

1. Which solution was most challenging to configure?
2. How would you adapt these solutions for a real deployment?
3. What other business problems could these solutions solve?

---

## 📚 References

- [Object Counting Docs](https://docs.ultralytics.com/guides/object-counting/)
- [Heatmaps Docs](https://docs.ultralytics.com/guides/heatmaps/)
- [Speed Estimation Docs](https://docs.ultralytics.com/guides/speed-estimation/)
- [Analytics Docs](https://docs.ultralytics.com/guides/analytics/)
- [Parking Management Docs](https://docs.ultralytics.com/guides/parking-management/)
- [Workouts Monitoring Docs](https://docs.ultralytics.com/guides/workouts-monitoring/)

---

**End of Lab**